In [9]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 4.6 MB/s eta 0:00:00


In [115]:
import time
import numpy as np
import gradio as gr
from openai import OpenAI
from pypdf import PdfReader
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [116]:
MODEL="gpt-5-nano"
SENTENSE_TRANSFORMER_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

openai = OpenAI(
    api_key=OPENAI_API_KEY,
)

embedding_model = SentenceTransformer(
    SENTENSE_TRANSFORMER_MODEL
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [121]:
SYSTEM_MESSAGE = """
You are an expert Resume Analysis and Candidate Screening Assistant.

Your task is to analyze resume content and extract information into the following structured sections:

Heading:
- Candidate Name
- Contact Information
- Match Percentage

1. Skills
   - Technical skills
   - Soft skills
   - Tools, frameworks, and technologies

2. Work Experience
   - Job titles
   - Companies
   - Employment duration
   - Key responsibilities and achievements

3. Education
   - Degrees and certifications
   - Institutions
   - Graduation dates

4. Projects (if available)
   - Project names
   - Technologies used
   - Key contributions and outcomes

5. Certifications (if available)

6. Hobbies and Interests (if available)

7. Additional Information
   - Languages
   - Awards
   - Volunteer work
   - Other relevant details

8. Candidate Summary
   - Brief professional overview
   - Years of experience
   - Primary expertise areas

9. Candidate Evaluation
   - Key strengths
   - Potential concerns or gaps
   - Overall suitability for the role

Instructions:
- Extract only information that is explicitly mentioned in the resume.
- Do not invent or assume details.
- Organize the output clearly under the appropriate sections.
- Highlight the candidate's most relevant qualifications and achievements.
- Provide a concise assessment to help recruiters identify the best candidates.
- Maintain a professional and objective tone throughout the analysis.
"""

In [122]:
def find_resume_match_percentage(resume_content , job_description):
  resume_embedding = embedding_model.encode([resume_content])
  job_description_embedding = embedding_model.encode([job_description])
  similarity = cosine_similarity(job_description_embedding , resume_embedding)[0][0]
  similarity_score = similarity * 100
  return float(similarity_score)


In [123]:
def ai_screening(resume_content , percentage_calculateion):
  ai_response = openai.chat.completions.create(
    model=MODEL,
    messages=[
      {
        "role": "system",
        "content": SYSTEM_MESSAGE
      },
      {
        "role": "user",
        "content": f"resume_content: {resume_content}\n\npercentage_calculateion: {percentage_calculateion}"
      }
    ]
  )
  return ai_response.choices[0].message.content

In [124]:
def upload_file(file , job_content):
  if file is None:
    return "Please upload a file."

  try:
    reader = PdfReader(file.name)
    text = ""
    for page in reader.pages:
      text += page.extract_text() + "\n"
      percentage_calculateion = find_resume_match_percentage(text, job_content)
      ai_generated_text = ai_screening(text , percentage_calculateion)
      for i in range(len(ai_generated_text)):
        time.sleep(0.1)
        yield ai_generated_text[:i+1]
      # return f"## Screening Content\n\nExtracted Text:\n\n```\n{text}\n```"
  except Exception as e:
    return f"Error reading file: {e}. Please ensure it is a valid PDF."


In [125]:
with gr.Blocks() as screening_form:
    gr.Markdown("# Upload Resume for AI Screening")
    with gr.Row():
      with gr.Column(scale=1):
        input_file = gr.File(label="Upload Resume")
        job_content = gr.Textbox(label="Job Description")
        submit_button = gr.Button("Submit" , variant="primary")
      with gr.Column(scale=1):
        screening_content = gr.Markdown("## Screening Content")

    submit_button.click(
        fn=upload_file,
        inputs=[input_file , job_content],
        outputs=screening_content
    )

In [126]:
screening_form.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20ea074c33f8617a5b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
